In [ ]:
# pylint: disable=wrong-import-position
# pylint: disable=wrong-import-order
# pylint: disable=invalid-name

# Building a Cognitive Agent - A Cogent
## From Decision Processes to Complete Agents

In the previous chapter, we explored how to implement core decision-making patterns using the `cognition` library. Now, we shift our focus toward assembling these patterns into a fully realized agent.

While the software industry often uses "agent" as a catch-all term for LLM orchestration or multi-step prompting pipelines, `cognition` grounds its design in classical AI agent science. Our framework is built directly upon the foundational **Perceive-Decide-Act** loop established in Russell & Norvig’s [*Artificial Intelligence: A Modern Approach (Chapter 2)*](https://people.eecs.berkeley.edu/~russell/aima1e/chapter02.pdf).

At its core, an agent is a computational entity that continuously cycles through three phases:

* **Perceive:** Gathers real-time telemetry and state from its environment through **sensors**.
* **Decide:** Processes incoming information to determine which action should be taken next.
* **Act:** Executes changes back onto the external environment using **actuators**.

---

> 💡 **The LLM connection: mapping the agent loop to large-model-based agents**
> 
> Even state-of-the-art conversational models like **ChatGPT**, **Gemini**, and **Claude** rely on this exact classical loop under the hood:
> 
> * **Perceive:** The model ingests the user prompt and context window as its primary sensory input.
> * **Decide:** The underlying transformer architecture processes this context through its attention layers to compute log-probabilities (`log_probs`) over target tokens.
> * **Act:** The model samples from those `log_probs` to output generated text back to the user or call tool APIs that augment the context. 

---

### `cognition` agents
Building upon this agent loop, the `cognition` library enables engineers to construct sophisticated agents that communicate, solve problems, make decisions, and learn. Rather than relying on a single model, you can seamlessly integrate [large language models](https://en.wikipedia.org/wiki/Large_language_model), [neural network inference](https://en.wikipedia.org/wiki/Neural_network_(machine_learning)), [symbolic reasoning](https://en.wikipedia.org/wiki/Symbolic_artificial_intelligence), and [knowledge graphs](https://en.wikipedia.org/wiki/Knowledge_graph) into a unified architecture.

`cognition` traces its lineage back to the [cognitive architecture](https://en.wikipedia.org/wiki/Cognitive_architecture) movement, most notably the [Soar](https://soar.eecs.umich.edu/) cognitive architecture. By blending decades of cognitive AI research with modern deep learning, it provides a stable framework for complex, long-running agents.

### Defining the "Cogent"

To distinguish basic prompt-chaining workflows from deeper, knowledge-rich agents built with our framework, we use a distinct term: **cogent** (*a cognitive agent*).

In this tutorial, you will set up, configure, and run your first working cogent.

---

## Setting up the Smart Home Simulation Environment

Before building a cogent, we need an environment for it to interact with. This tutorial is designed with an virtual environment example that simulates a smart home. It is included with the tutorial. This enviroment builds upon Flask's implementation of RESTful API. 

This mock environment uses a Terminal User Interface (TUI) powered by `textual`, allowing you to visually inspect and manipulate device states alongside your agent.

### Step 1: install and launch the simulation

Open a separate terminal window, navigate to your workspace, and run the following commands to install the required dependencies and start the interactive simulation:

```bash
# Install the environment prerequisites
pip install -r smart_home/requirements.txt

# Launch the interactive Textual simulation application
python smart_home/smart_home.py
```

Spend a brief moment clicking or using keyboard controls in the interactive TUI app to turn virtual switches on and off to familiarize yourself with its live behaviors.

### Step 2: Establish a Connection
In your Python environment, verify the connection to the running simulation:

In [ ]:
"""Demo smart home agent built with polycog cognition."""

### add the location of cognition/examples/smart_home to your PYTHONPATH
from smart_home.client import SmartHomeClient

client = SmartHomeClient()
try:
    client.connect()
    print("Successfully connected to Smart Home Simulation!")
except ConnectionError as exc:
    raise SystemExit(
        "Connection failed. Ensure the simulation app is running in another terminal.\n"
        f"Details: {exc}"
    ) from exc

### Step 3: Inspect Live Observations
The `.observe()` method returns device states from the environment. This method will be used by our cogent to read data from the environment.

Try toggling the physical lamp toggle switch inside the TUI application window, and rerun this code to see the observation state reflect changes dynamically:

In [ ]:
# Fetch current sensor telemetry
response = client.observe()
print(response)

### Step 4: Actuate Programmatically 
In addition to watching the environment, our cogent will need to manipulate it via actuators. You can test programmatic actuation manually from your workspace using the `.actuate()` method:

In [ ]:
# Programmatically turn the lamp off, then back on
client.actuate("lamp", "turn_off")
client.actuate("lamp", "turn_on")

# Verify the resulting state
print(client.observe())

Now that your simulation is wired up and your notebook can successfully read and actuate devices, you are ready to build your first cogent!

----

## Building a Cogent

With the simulation connected, we can now build a cognitive agent that links device sensing and actuation directly to the decision process designed in Tutorial 1.

### Step 1: Configure State

First, let's configure our internal state. Unlike Tutorial 1, we don't need to manually track physical device states internally; the simulation environment handles physical state for us. We only maintain higher-level operational context, such as the active time mode.

In [ ]:
from dataclasses import dataclass
from enum import Enum, auto


class Mode(Enum):
    """Operating mode for the home assistant agent."""

    DAY = auto()
    EVENING = auto()
    MIDNIGHT = auto()


@dataclass
class HomeState:
    """Internal state maintained by the agent."""

    mode: Mode = Mode.DAY

### Step 2. Defining data contracts.
The smart home environment responds to the agent's API calls using JSON strings. However, directly parsing and manipulating raw strings is error-prone. To build a robust system, we define explicit data contracts to enforce structure and type safety:
- `Enum`: For representing discrete device states and control signals.
- `dataclass (frozen=True)`: Defines immutable, structured payloads passed between the agent and the environment.

In [ ]:
from enum import StrEnum


class DeviceState(StrEnum):
    """Device state values reported by the smart home simulation environment."""

    ON = "on"
    OFF = "off"


class ControlSignal(StrEnum):
    """Control commands sent to the smart home simulation environment."""

    TURN_ON = "turn_on"
    TURN_OFF = "turn_off"


@dataclass(frozen=True)
class Observation:
    """Immutable snapshot of environmental readings passed to the agent."""

    lamp: DeviceState


@dataclass(frozen=True)
class Action:
    """Immutable action command payload emitted by the agent."""

    lamp: ControlSignal

### Step 3: instantiate the Cogent

In the `cognition` library, the `Cogent` class manages the agent's interaction loop following the **Perceive–Decide–Act** pattern:

1. **Perceive (`Sensor`)**: Polls external telemetry and writes structured observations into input (`IOContainer.i`).
2. **Decide (`DecisionProcess`)**: Evaluates incoming observations and current state using `Operators`. Operators read from `IOContainer.i`, update internal `State`, reason, and stage actions into output (`IOContainer.o`).
3. **Act (`Actuator`)**: Reads pending actions from `IOContainer.o` and dispatches commands back to the external environment.

As an agent designer, your primary role is to define the custom logic for each of these components. Let’s set up the basic `Cogent` instance first before implementing its specific functionality.

In [ ]:
from cognition import Cogent, DecisionProcess

# Set initial operational state
state = HomeState(mode=Mode.MIDNIGHT)

# Create the decision process bound to agent state
dp = DecisionProcess[HomeState](lambda: state)

# Instantiate the Cogent
assistant = Cogent[DecisionProcess[HomeState]](decision_process=dp)

### Step 4: implement and register Sensors and Actuators
Sensors and actuators serve as the I/O bridge between your agent and the external environment:

- `Sensor`: Ingests environmental telemetry and writes structured observations into `IOContainer.i`. Each sensor requires a unique `name` property (which `Operator`s use to look up data) and overrides the `sense()` method to query the environment API and return a payload.

- `Actuator`: Handles outbound control actions staged in `IOContainer.o`. Each actuator requires a unique `name` property and overrides the `actuate()` method to send action commands back to the environment API.

> Design Pattern: Frequently, a single class can implement both `Sensor` and `Actuator` generic interfaces to streamline bi-directional communication with a given device or API endpoint.

Below is a composite interface that handles both telemetry polling and command dispatching, followed by registering it with the Cogent instance:

In [ ]:
from cognition import Actuator, Sensor


class SmartHomeDevices(Sensor[Observation], Actuator[Action, None]):
    """Composite interface providing both sensing and actuation capabilities."""

    @property
    def name(self) -> str:
        return "devices"

    def sense(self) -> Observation:
        """Poll the environment API and parse state into an Observation contract."""
        res = client.observe()
        return Observation(lamp=DeviceState(res["lamp"]))

    def actuate(self, param: Action) -> None:
        """Dispatch control commands to the target environment API."""
        client.actuate("lamp", param.lamp)


# Instantiate composite device interface and attach to cogent
devices = SmartHomeDevices()
assistant.add_sensor(devices).add_actuator(devices)

### Step 5: implement and register decision operators

With our SmartHomeDevices sensor streaming telemetry into `IOContainer.i`, we can now implement the Decide phase of the Perceive–Decide–Act loop by adapting the `TurnLightOff` `Operator` from Tutorial 1.  

- `can_perform(state, io)`: Inspects internal `State` and `Observation` on `io.i`to determine if an action should be triggered.
- `perform(state, io)`: Executes the decision by staging an `Action` payload into `io.o` for the actuator to dispatch during the Act phase.


In [ ]:
from typing import cast

from cognition import IOContainer, Operator


# pylint: disable=redefined-outer-name
class TurnLightOff(Operator[HomeState]):
    """
    WHEN: mode is MIDNIGHT and lamp is ON.
    THEN: turn the lamp off.
    """

    def can_perform(self, state: HomeState, io: IOContainer) -> bool:
        """Propose action when mode is MIDNIGHT and the lamp is ON."""
        observation = cast(Observation, io.i.devices)
        return state.mode is Mode.MIDNIGHT and observation.lamp == DeviceState.ON

    def perform(self, state: HomeState, io: IOContainer) -> None:
        """Emit turn off signal to actuator container."""
        io.o.devices(Action(lamp=ControlSignal.TURN_OFF))
        print("--> [Action Triggered] Mode is MIDNIGHT. Turning light off.")


# Register operator with decision process
assistant.dp.add_operator(TurnLightOff("turn_light_off", terminal=True))

### Step 6: run the cogent

With our components fully configured: `Observation` and `Action` data contracts established, `devices` sensor and actuator attached (Section 4), and the `TurnLightOff` decision operator registered, we can now start the perpetual **Perceive–Decide–Act** runtime loop..

To run the agent continuously in production or local development, we configure two key policy predicates:

- Loop Termination Predicate (`run_forever`): Controls the execution lifespan of the `Cogent` instance. 
- Error Gate Predicate (`gate_no_potential_actions`): Handles execution errors and idle states. When no operator's `can_perform()` condition is met, the engine raises `DecisionProcessErrorMessage.NO_PROPOSAL`. Instead of crashing, our error gate catches this signal, pauses briefly, and allows the loop to poll continuously for new sensory inputs.

In [ ]:
import time
from typing import Any

from cognition import DecisionProcessErrorMessage


def run_cogent(cogent: Cogent[DecisionProcess[Any]]):
    """Run the cogent's perpetual Perceive-Decide-Act loop."""

    def run_forever(_cogent) -> bool:
        """Keep the execution loop running continuously."""
        return True

    def gate_no_potential_actions(err: DecisionProcessErrorMessage, _cogent) -> bool:
        """
        Handle decision process errors.
        Allows the agent to remain idle when no actions are proposed, while pausing
        briefly to prevent tight CPU looping.
        """
        time.sleep(0.5)
        if err is DecisionProcessErrorMessage.NO_PROPOSAL:
            return True  # Suppress error and continue execution loop
        raise RuntimeError(err)  # Re-raise unexpected critical errors

    print(
        "Cogent running. Click 'Interrupt Kernel' in Jupyter or press Ctrl+C in terminal to stop.\n"
    )
    try:
        # Pass the loop predicate and error handling policy to the Cogent instance
        cogent(run_forever, dp_err_p=gate_no_potential_actions)
    except KeyboardInterrupt:
        print("\nExecution interrupted.")

In [ ]:
run_cogent(assistant)

#### How the Runtime Cycle Operates
When run_cogent() executes, smart_home_assistant continuously cycles through the architecture established across previous sections:

1. **Perceive**: Calls `devices.sense()` to update `IOContainer.i.devices` with the latest environmental observation.

2. **Evaluate Operators**: Checks `TurnLightOff.can_perform()` against the new observation and internal state.
- If conditions are met, `TurnLightOff.perform()` stages the action into IOContainer.o.devices (Section 5).
- If conditions are not met, `gate_no_potential_actions` intercepts `NO_PROPOSAL`, sleeps for 0.5s, goes to the **Perceive** step again. 

3. **Act**: Triggers `devices.actuate()`, sending the `TURN_OFF` payload back to the external environment API.

### Step 7: observe cogent behavior and explroe UX trade-offs
Once `run_cogent()` is active, test the interaction loop by interacting with your simulation terminal UI (TUI).

> ⚠️ Key runtime observation > expected behavior: The agent runs continuously in the background. As soon as you manually toggle the switch ON inside the simulation TUI, the agent immediately detects the state change and forces it back OFF.

#### Is this a good user experience?



## Developer exercise: improving cogent behavior with user modeling

Our current cogent has a flawed real-world user experience. If a resident wakes up at 2 AM to get a glass of water, the agent immediately turns off the lamp, plunging them back into total darkness.

To make our cogent smarter, we must account for human behavior. A resident getting up at night needs light briefly. Instead of shutting the light off instantly, an elegant solution is to introduce a **user model** assumption: midnight activity should remain on for a configurable duration (e.g., $N$ seconds). If the light remains on past this window, the cogent then steps in to cut the power.

Let's implement this improved behavior in four steps.

### Step 1: Extend State to Track a Timer

Extend internal state with a `timer_expires_at` attribute to keep track of countdown expiry.

In [ ]:
@dataclass
class NewState(HomeState):
    """Internal agent state to track countdown timer."""

    timer_expires_at: float = 0.0

### Step 2: implement `StartTimer` operator

Write an operator that detects when a light turns on at midnight and initializes a 5-second timer if one isn't active already.

In [ ]:
class StartTimer(Operator[NewState]):
    """
    WHEN: mode is MIDNIGHT, lamp is ON, and no timer is running.
    THEN: start a 5-second countdown timer.
    """

    def can_perform(self, state: NewState, io: IOContainer) -> bool:
        observation = cast(Observation, io.i.devices)
        return (
            state.mode is Mode.MIDNIGHT
            and observation.lamp is DeviceState.ON
            and state.timer_expires_at == 0.0
        )

    def perform(self, state: NewState, io: IOContainer) -> None:
        state.timer_expires_at = time.monotonic() + 5.0
        print("--> [Timer Started] 5-second countdown initialized.")

### Step 3: update `TurnLightOff` operator to respect elapsed time

Update `TurnLightOff` so it triggers only after the 5-second timer has fully elapsed.

In [ ]:
class TurnLightOffNew(Operator[NewState]):
    """
    WHEN: mode is MIDNIGHT, lamp is ON, timer is active, and timer has expired.
    THEN: turn off the lamp and reset the timer.
    """

    def can_perform(self, state: NewState, io: IOContainer) -> bool:
        observation = cast(Observation, io.i.devices)
        return (
            state.mode is Mode.MIDNIGHT
            and observation.lamp == DeviceState.ON
            and state.timer_expires_at != 0.0
            and time.monotonic() >= state.timer_expires_at
        )

    def perform(self, state: NewState, io: IOContainer) -> None:
        io.o.devices(Action(lamp=ControlSignal.TURN_OFF))
        state.timer_expires_at = 0.0
        print("--> [Action Triggered] Timer expired. Turning light off.")

### Step 4: initialize and test the user-aware cogent

Instantiate the new decision process and cogent, register both operators, and run the execution loop to observe the improved behavior. Notice the `terminal=True` flag in `add_operator`. This flag is another way of writing a terminal check - it tells the `cognition` engine to terminate the decision process after applying this operator.

In [ ]:
# 1. Initialize new state and decision process
new_state = NewState(mode=Mode.MIDNIGHT)
new_dp: DecisionProcess[NewState] = DecisionProcess(lambda: new_state)

# 2. Instantiate cogent
new_assistant = Cogent[DecisionProcess[NewState]](decision_process=new_dp)
new_assistant.add_sensor(devices).add_actuator(devices)

# 3. Add operators in evaluation order
new_assistant.dp.add_operator(StartTimer("start_timer", terminal=True))
new_assistant.dp.add_operator(TurnLightOffNew("turn_light_off", terminal=True))

# 4. Run cogent
run_cogent(new_assistant)

Can you think of any other issue? What happens when the resident shuts the light off themselves before the timer expires? 

Check out the full implementation at [smart_home_assistant_v2.py](smart_home_assistant_v2.py).